# पाठ ०४ - उपकरण प्रयोग डिजाइन ढाँचा

यस पाठमा तपाईं Microsoft Agent Framework (Python) प्रयोग गरेर AI एजेन्टहरुका लागि **उपकरण प्रयोग** डिजाईन ढाँचाको बारेमा सिक्नु हुनेछ। हामीले समेटेका छौं:

- `@tool` डेकोरेटर र टाइप गरिएको प्यारामिटरहरू सहित फंक्शन उपकरणहरू परिभाषित गर्ने
- मोडेललाई थाहा होस् भनेर प्रत्येक उपकरणले के गर्छ भन्ने उपकरण स्किमाहरू प्रदान गर्ने
- `approval_mode` सँग उपकरण कार्यान्वयन नियन्त्रण गर्ने
- Pydantic मोडेलहरू र `response_format` मार्फत **संरचित परिणाम** फिर्ता गर्ने

परिदृश्य एक **यात्रा बुकिंग एजेन्ट** हो जुन गन्तव्यहरू खोज्न, उपलब्धता जाँच गर्न, र उडान जानकारी प्राप्त गर्न सक्छ।


## सेटअप


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## @tool डेकोरेटरसँग उपकरणहरू परिभाषित गर्दै

`@tool` डेकोरेटर साधारण Python फंक्शनलाई एउटा उपकरणमा परिवर्तन गर्छ जुन एजेन्टले कल गर्न सक्छ।
प्रमुख बुँदाहरू:

- **डोकरिंग** उपकरण वर्णन बनिन्छ जुन मोडेलले देख्छ।
- **टाइप एनोटेसनहरू** (वर्णनसहित `Annotated` सहित) उपकरण स्कीमा परिभाषित गर्छन्।
- `approval_mode` ले नियन्त्रण गर्छ कि प्रयोगकर्ताले प्रत्येक कल कार्यान्वयन हुनु अघि स्वीकृति दिनुपर्छ कि पर्दैन।


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## धेरै उपकरणहरूसँग एजेन्ट सिर्जना गर्न

ग्राहकलाई सबै तीन उपकरणहरू पठाउनुहोस् ताकि मोडेलले प्रयोगकर्ताको प्रश्नको उत्तर दिन आवश्यक पर्ने जुनसुकै उपकरणहरूलाई आह्वान गर्न सकोस्।


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = await provider.create_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## उपकरणहरूसँग संरचित आउटपुट

`response_format` लाई Pydantic मोडेल मा सेट गरेर, एजेन्टलाई स्वतन्त्र पाठको सट्टा राम्रो-टाइप गरिएको JSON वस्तु फिर्ता गर्न बाध्य पारिन्छ। जब तल्लो तहको कोडले परिणामलाई प्रोग्राम्याटिक रूपमा उपभोग गर्न आवश्यक पर्छ, तब यो उपयोगी हुन्छ।


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = await provider.create_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## उपकरण अनुमोदन ढाँचा

`@tool` मा रहेको `approval_mode` प्यारामिटरले उपकरण कलहरू कार्यान्वयन गर्नु अघि मानव अनुमोदन आवश्यक छ कि छैन भनेर नियन्त्रण गर्छ:

| मोड | व्यवहार |
|---|---|
| `"never_require"` | उपकरण स्वचालित रूपमा चल्छ — प्रयोगकर्ताको पुष्टिकरण आवश्यक छैन। |
| `"always_require"` | प्रत्येक कल कार्यान्वयन हुनु अघि प्रयोगकर्ताले अनुमोदन गर्नुपर्छ। |

तर्फ असर पार्ने उपकरणहरूको लागि (जस्तै: उडान बुक गर्नु, क्रेडिट कार्ड चार्ज गर्नु) `"always_require"` प्रयोग गर्नुहोस् ताकि मानव प्रक्रिया भित्रै रहोस्।


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## Summary

यस पाठमा तपाईंले सिक्नुभयो कसरी:

1. टाइप गरिएको प्यारामिटर र उपकरण स्कीमाको रूपमा सेवा गर्ने डकस्ट्रिङहरू सहित `@tool` डेकोरेटर प्रयोग गरी **उपकरणहरू परिभाषित गर्ने**।
2. जटिल क्वेरीहरूको जवाफ दिन एजेन्टलाई शृंखलाबद्ध रूपमा कल गर्न सक्ने गरी **धेरै उपकरणहरू संयोजन गर्ने**।
3. `response_format` को रूपमा Pydantic मोडल पास गरेर **संरचित आउटपुट फिर्ता गर्ने**।
4. संवेदनशील अपरेसनको लागि मान्छेलाई नियन्त्रणमा राख्न **`approval_mode` सँग उपकरण स्वीकृति नियन्त्रण गर्ने**।

यी ढाँचाहरू विश्वसनीय, उत्पादन-तयार एजेन्टहरू निर्माण गर्नको लागि आधार हुन् जो बाह्य प्रणालीहरूसँग सुरक्षित रूपमा अन्तरक्रिया गर्न सक्छन्।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
यो कागजात AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) को प्रयोग गरी अनुवाद गरिएको हो। हामी शुद्धताका लागि प्रयासरत छौं, तर कृपया ध्यान दिनुहोस् कि स्वचालित अनुवादमा त्रुटि वा अशुद्धता हुन सक्दछ। मौलिक कागजात यसको मूल भाषामा नै अधिकारिक स्रोतको रूपमा लिनु पर्छ। महत्वपूर्ण जानकारीको लागि व्यावसायिक मानव अनुवाद सिफारिस गरिन्छ। यस अनुवादको प्रयोगबाट उत्पन्न भएको कुनै पनि गलतफहमी वा गलत व्याख्याको लागि हामी जिम्मेवार हुँदैनौं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
